Librerias

In [17]:
!pip install -q kaggle
!pip install -q pillow
!pip install -q pandas
!pip install -q matplotlib
!pip install -q numpy
!pip install imagehash --quiet


[notice] A new release of pip is available: 26.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 26.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 26.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 26.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 26.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 26.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [18]:
import pandas as pd
import numpy as np
import os
import imagehash
import matplotlib.pyplot as plt
from PIL import Image, UnidentifiedImageError
from procesamiento import dividir_train_val_test, calcular_hash, marcar_duplicado_verificado, son_realmente_duplicados
from sklearn.preprocessing import LabelEncoder




Cosas necesarias para tener el  train test

In [19]:
N_POR_CLASE = 600
SEMILLA = 42
TAM_OBJETIVO = (64, 64)
np.random.seed(SEMILLA)


RUTA_DATASET = r"archive/asl_alphabet_train/asl_alphabet_train"

datos_img = []

for clase in sorted(os.listdir(RUTA_DATASET)):
    carpeta = os.path.join(RUTA_DATASET, clase)
    if not os.path.isdir(carpeta):
        continue

    for archivo in os.listdir(carpeta):
        ruta = os.path.join(carpeta, archivo)
        try:
            imagen = Image.open(ruta)
            ancho, alto = imagen.size
            datos_img.append({
                "clase": clase,
                "archivo": archivo,
                "ruta": ruta,
                "ancho": ancho,
                "alto": alto,
                "modo": imagen.mode,
                "formato": imagen.format
            })
        except Exception as e:
            print(f"No se pudo abrir {ruta}")

datos_img = pd.DataFrame(datos_img)

print("Cantidad total de imágenes:")
print(len(datos_img))

datos_img.head()

submuestra = (
    datos_img.groupby("clase", group_keys=False)
    .apply(lambda x: x.sample(n=min(N_POR_CLASE, len(x)), random_state=SEMILLA))
    .reset_index(drop=True)
)

    
submuestra["phash"] = submuestra["ruta"].apply(lambda r: calcular_hash(r, hash_size=16))


Cantidad total de imágenes:
87000


C:\Users\Usuario Preinstalado\AppData\Local\Temp\ipykernel_2636\867506301.py:42: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(n=min(N_POR_CLASE, len(x)), random_state=SEMILLA))


duplicados y transformaciones

In [20]:
def cargar_y_preprocesar(df, tam=TAM_OBJETIVO):
    """Carga imágenes desde 'ruta', redimensiona y normaliza a [0, 1]."""
    imagenes = []
    for ruta in df["ruta"]:
        img = Image.open(ruta).convert("RGB").resize(tam)
        arr = np.array(img, dtype=np.float32) / 255.0
        imagenes.append(arr)
    X = np.stack(imagenes)
    y = df["clase"].values
    return X, y

duplicados = submuestra[submuestra.duplicated(subset="phash", keep=False)]
print(f"\nImágenes con hash repetido: {len(duplicados)} de {len(submuestra)}")

indices_duplicados_reales = []
for phash_val, grupo in duplicados.groupby("phash"):
    indices_duplicados_reales.extend(marcar_duplicado_verificado(grupo))

submuestra_limpia = submuestra.drop(index=indices_duplicados_reales).reset_index(drop=True)
print(f"Duplicados confirmados y eliminados: {len(indices_duplicados_reales)}")
print(f"Submuestra final: {len(submuestra_limpia)} imágenes")

# Verificar que la limpieza no haya desbalanceado alguna clase
print("\nDistribución de clases tras limpieza:")
print(submuestra_limpia["clase"].value_counts().sort_values())


#SPLIT TRAIN / VAL / TEST (70/15/15, estratificado)
train_df, val_df, test_df = dividir_train_val_test(
    df=submuestra_limpia,
    col_clase="clase",
    semilla=SEMILLA
)


# PREPROCESAMIENTO FINAL: resize + normalización -> arrays listos para modelar
X_train, y_train_raw = cargar_y_preprocesar(train_df)
X_val, y_val_raw = cargar_y_preprocesar(val_df)
X_test, y_test_raw = cargar_y_preprocesar(test_df)

# Codificación de etiquetas (texto -> enteros); se ajusta SOLO con train
codificador = LabelEncoder()
codificador.fit(y_train_raw)

y_train = codificador.transform(y_train_raw)
y_val = codificador.transform(y_val_raw)
y_test = codificador.transform(y_test_raw)

print(f"\nX_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_val:   {X_val.shape}, y_val:   {y_val.shape}")
print(f"X_test:  {X_test.shape}, y_test:  {y_test.shape}")
print(f"Rango de X_train tras normalizar: [{X_train.min():.2f}, {X_train.max():.2f}]")
print(f"Clases codificadas: {dict(zip(codificador.classes_, range(len(codificador.classes_))))}")



Imágenes con hash repetido: 22 de 17400
Duplicados confirmados y eliminados: 9
Submuestra final: 17391 imágenes

Distribución de clases tras limpieza:
clase
space      599
I          599
K          599
M          599
B          599
U          599
O          599
Y          599
E          599
W          600
C          600
D          600
nothing    600
F          600
A          600
del        600
J          600
Z          600
L          600
N          600
X          600
P          600
Q          600
R          600
S          600
T          600
V          600
H          600
G          600
Name: count, dtype: int64
Train: 12173 (70.0%) | Val: 2609 (15.0%) | Test: 2609 (15.0%)

X_train: (12173, 64, 64, 3), y_train: (12173,)
X_val:   (2609, 64, 64, 3), y_val:   (2609,)
X_test:  (2609, 64, 64, 3), y_test:  (2609,)
Rango de X_train tras normalizar: [0.00, 1.00]
Clases codificadas: {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4, 'F': 5, 'G': 6, 'H': 7, 'I': 8, 'J': 9, 'K': 10, 'L': 11, 'M': 12, 'N': 1